# SalesInsight-PY

## Criando o Dataset de vendas

In [25]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
 # Função abaixo foi usado para gerar um dataset randomico.
'''def gerar_dataset_vendas(n_registros=1500, seed=42):
    """Gera um dataset sintetico de vendas com dados sujos."""
    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",
                "Teclado", "Mouse", "Headset"]
    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",
                  "Tablet": "Celulares", "Monitor": "Computadores",
                  "Teclado": "Perifericos", "Mouse": "Perifericos",
                  "Headset": "Perifericos"}
    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,
              "Monitor": 1200, "Teclado": 250, "Mouse": 120,
              "Headset": 350}
    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]

    data_inicio = datetime(2025, 1, 1)
    dados = []

    for i in range(n_registros):
        produto = random.choice(produtos)
        categoria = categorias[produto]
        quantidade = random.randint(1, 10)
        preco = round(precos[produto] * random.uniform(0.85, 1.15), 2)
        data = data_inicio + timedelta(days=random.randint(0, 364))
        data_txt = data.strftime("%Y-%m-%d")
        cliente = f"Cliente_{random.randint(1, 50):03d}"

        # --- sujeira proposital para a etapa de limpeza --- 
        if random.random() < 0.03:
        	data_txt = "DATA INVALIDA"       	# data invalida

        dados.append({
            "id_venda": i + 1,
            "data_venda": data_txt,
            "cliente": cliente,
            "produto": produto,
            "categoria": categoria,
            "regiao": random.choice(regioes),
            "quantidade": quantidade,
            "preco_unitario": preco,
        })

    return pd.DataFrame(dados)


# Gerar e salvar o CSV bruto
df_bruto = gerar_dataset_vendas()
df_bruto.to_csv("vendas.csv", index=False)
print(f"Dataset gerado com {len(df_bruto)} registros.")
print(df_bruto.head())'''


'def gerar_dataset_vendas(n_registros=1500, seed=42):\n    """Gera um dataset sintetico de vendas com dados sujos."""\n    random.seed(seed)\n    np.random.seed(seed)\n\n    produtos = ["Notebook", "Smartphone", "Tablet", "Monitor",\n                "Teclado", "Mouse", "Headset"]\n    categorias = {"Notebook": "Computadores", "Smartphone": "Celulares",\n                  "Tablet": "Celulares", "Monitor": "Computadores",\n                  "Teclado": "Perifericos", "Mouse": "Perifericos",\n                  "Headset": "Perifericos"}\n    precos = {"Notebook": 3500, "Smartphone": 2200, "Tablet": 1800,\n              "Monitor": 1200, "Teclado": 250, "Mouse": 120,\n              "Headset": 350}\n    regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]\n\n    data_inicio = datetime(2025, 1, 1)\n    dados = []\n\n    for i in range(n_registros):\n        produto = random.choice(produtos)\n        categoria = categorias[produto]\n        quantidade = random.randint(1, 10)\n      

## Inspecionar e Descrever os Dados

In [26]:
import pandas as pd
dados = pd.read_csv("vendas.csv")
def inspecionar_dados(df):
    print("=== INSPECAO INICIAL DO DATASET ===")
    print("Shape:", df.shape)
    print("Colunas:\n", list(df.columns))
    print("Tipos de dados:\n", df.dtypes)
    print("Valores nulos por coluna:\n", df.isnull().sum())
    print("Primeiros registros:\n", df.head)
    return df

inspec_inicial = inspecionar_dados(dados)

=== INSPECAO INICIAL DO DATASET ===
Shape: (1500, 8)
Colunas:
 ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']
Tipos de dados:
 id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade          int64
preco_unitario    float64
dtype: object
Valores nulos por coluna:
 id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64
Primeiros registros:
 <bound method NDFrame.head of       id_venda     data_venda      cliente     produto     categoria  \
0            1     2025-05-21  Cliente_016       Mouse   Perifericos   
1            2  DATA INVALIDA  Cliente_003       Mouse   Perifericos   
2            3     2025-10-15  Cliente_013  Smartphone     Celulares   
3            4     2025-05-23  Cliente_001     Monitor

## Tratar os Dados (datetime)

In [ ]:
def tratar_dados(df):
     """  Trata o DataFrame de vendas.  Retorna: (df_limpo, relatorio), onde relatorio e um dicionario  
               com as contagens de registros iniciais, convertidas e finais.  """
     df = df.copy()
     # Remover espacos extras nas colunas de texto
     df.columns = df.columns.str.strip()

     registros_iniciais = len(df)
     
     # Converter data_venda e descartar datas invalidas
     df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
     registros_excluidos = df["data_venda"].isna().sum()
     df_limpos = df.dropna(subset=["data_venda"]).copy()
     
     # Ajustar os tipos numericos
     df_limpos["quantidade"] = df_limpos["quantidade"].astype(int)
     df_limpos["preco_unitario"] = df_limpos["preco_unitario"].astype(float)
     # print(df[["quantidade", "preco_unitario"]].dtypes)
     # Exibir os tipos numericos de quantidade e preco_unitario.
     
     # Montar e imprimir o relatorio de limpeza
     relatorio = {
          "registros_iniciais": registros_iniciais,
          "registros_excluidos": int(registros_excluidos),
          "registros_finais": len(df_limpos),
     }
     print(relatorio)

     return df_limpos, relatorio

dados_limpos, relatorio = tratar_dados(dados)


{'registros_iniciais': 1500, 'registros_excluidos': 51, 'registros_finais': 1449}


## Criar Colunas Derivadas com Transformações Condicionais

In [ ]:

import numpy as np
import pandas as pd

# Criar uma cópia do dataset limpo
dados_limpos = dados_limpos.copy()

# Criando receita total: quantidade x preço unitário
dados_limpos["receita_total"] = (dados_limpos["quantidade"] * dados_limpos["preco_unitario"])

# Criando mes numericos
dados_limpos["mes"] = dados_limpos["data_venda"].dt.month

# Nome do mês em português
meses_pt = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}

dados_limpos["mes_nome"] = dados_limpos["mes"].map(meses_pt)

# Trimestre no formato Q1 a Q4
dados_limpos["trimestre"] = ("Q" + dados_limpos["data_venda"].dt.quarter.astype(str))

# Ano da venda
dados_limpos["ano"] = dados_limpos["data_venda"].dt.year

# Classificação condicional da receita por item
dados_limpos["faixa_receita_item"] = "Baixa"

dados_limpos.loc[dados_limpos["receita_total"] >= 500, "faixa_receita_item"] = "Média"

dados_limpos.loc[dados_limpos["receita_total"] >= 1000, "faixa_receita_item"] = "Alta"

# Exibir os resultados
print("Colunas derivadas criadas com sucesso!")
print(dados_limpos[["data_venda","quantidade","preco_unitario","receita_total",
        "mes","mes_nome","trimestre","ano","faixa_receita_item"]].head())

# Conferir a quantidade de registros por faixa
print("\nDistribuição por faixa de receita:")
print(dados_limpos["faixa_receita_item"].value_counts())


Colunas derivadas criadas com sucesso!
  data_venda  quantidade  preco_unitario  receita_total  mes mes_nome  \
0 2025-05-21           2          102.90         205.80    5     Maio   
2 2025-10-15           9         2267.33       20405.97   10  Outubro   
3 2025-05-23           4         1181.72        4726.88    5     Maio   
4 2025-03-21           7          114.25         799.75    3    Março   
5 2025-07-03           2         3373.92        6747.84    7    Julho   

  trimestre   ano faixa_receita_item  
0        Q2  2025              Baixa  
2        Q4  2025               Alta  
3        Q2  2025               Alta  
4        Q1  2025              Média  
5        Q3  2025               Alta  

Distribuição por faixa de receita:
faixa_receita_item
Alta     1153
Média     164
Baixa     132
Name: count, dtype: int64
